In [1]:

import json
from pathlib import Path
from collections import Counter

# -------- CONFIGURATION --------
INPUT_FILE = "./dataset/V2B/V2B.jsonl"
OUTPUT_DIR = "./split_by_turn_count"

# Ranges from your screenshot
CATEGORIES = {
    "extra_small_1_2": (1, 2),
    "small_3_6": (3, 6),
    "medium_7_15": (7, 15),
    "large_16_30": (16, 30),
    "extra_large_31_plus": (31, float("inf")),
}

# -------- HELPERS --------
def get_conversation(record):
    """Find the conversation text/list in a JSON record."""
    if isinstance(record, list):
        return record

    if not isinstance(record, dict):
        return None

    for key in ("conversation", "messages", "turns", "dialogue"):
        value = record.get(key)
        if isinstance(value, (list, str)):
            return value

    # Support nested input format
    value = record.get("input")
    if isinstance(value, dict):
        for key in ("conversation", "messages", "turns", "dialogue"):
            if isinstance(value.get(key), (list, str)):
                return value[key]

    return None


def count_turns(conversation):
    """Count individual messages/turns."""
    if isinstance(conversation, list):
        return len(conversation)

    if isinstance(conversation, str):
        # Each non-empty line is treated as one message/turn.
        return sum(
            1 for line in conversation.splitlines() if line.strip()
        )

    return 0


def get_category(turn_count):
    for category, (minimum, maximum) in CATEGORIES.items():
        if minimum <= turn_count <= maximum:
            return category

    return "other"


# -------- PROCESS DATASET --------
input_path = Path(INPUT_FILE)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

counts = Counter()
invalid_records = 0
total_records = 0

# Open separate output files
output_files = {
    category: open(
        output_dir / f"{category}.jsonl",
        "w",
        encoding="utf-8"
    )
    for category in [*CATEGORIES.keys(), "other"]
}

try:
    with input_path.open("r", encoding="utf-8") as source:
        for line_number, line in enumerate(source, start=1):
            if not line.strip():
                continue

            total_records += 1

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                invalid_records += 1
                continue

            conversation = get_conversation(record)

            if conversation is None:
                invalid_records += 1
                continue

            turn_count = count_turns(conversation)

            # Empty conversations are excluded from turn categories.
            if turn_count == 0:
                invalid_records += 1
                continue

            category = get_category(turn_count)

            # Preserve original record without changing its structure.
            output_files[category].write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

            counts[category] += 1

finally:
    for file in output_files.values():
        file.close()


# -------- REPORT --------
valid_records = sum(counts.values())

print("\n" + "=" * 55)
print("CONVERSATION TURN COUNT")
print("=" * 55)
print(f"Total records read : {total_records}")
print(f"Valid records      : {valid_records}")
print(f"Invalid/empty      : {invalid_records}")
print("-" * 55)

for category in [*CATEGORIES.keys(), "other"]:
    count = counts[category]
    percentage = (
        count / valid_records * 100 if valid_records else 0
    )
    print(f"{category:25} : {count:8} ({percentage:6.2f}%)")

print("=" * 55)
print(f"Output folder: {output_dir.resolve()}")



CONVERSATION TURN COUNT
Total records read : 74110
Valid records      : 74110
Invalid/empty      : 0
-------------------------------------------------------
extra_small_1_2           :       89 (  0.12%)
small_3_6                 :     2054 (  2.77%)
medium_7_15               :    26636 ( 35.94%)
large_16_30               :    40095 ( 54.10%)
extra_large_31_plus       :     5236 (  7.07%)
other                     :        0 (  0.00%)
Output folder: G:\V1-V16-dataset\split_by_turn_count
